# A5: CNN Architecture com Conv2D, MaxPooling, Flatten e Dense

Este notebook demonstra a construção de uma CNN (Convolutional Neural Network) com a arquitetura:

**Input → Conv2D (32 filtros) → MaxPooling → Conv2D (64 filtros) → MaxPooling → Flatten → Dense → Output**

Documentaremos todos os hiperparâmetros e mostraremos exemplos práticos.

## 1. Importar Bibliotecas Necessárias

Vamos importar TensorFlow, Keras e outras bibliotecas essenciais para construir e treinar modelos CNN.

In [7]:
%pip install -q tensorflowimport sysimport osimport numpy as npimport matplotlib.pyplot as pltimport tensorflow as tffrom tensorflow.keras import Sequential, layersfrom tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropoutfrom tensorflow.keras.optimizers import Adam# Configurar matplotlibplt.style.use('seaborn-v0_8-darkgrid')%matplotlib inline# Adicionar src ao path com path absolutoworkspace_path = r'c:\Users\Inteli\Desktop\g01\src'if workspace_path not in sys.path:    sys.path.insert(0, workspace_path)try:    from models.cnn_builder import build_cnn_model, get_model_architecture_summary    print("- cnn_builder importado com sucesso!")except ImportError as e:    print(f"⚠ Erro ao importar cnn_builder: {e}")print(f"- TensorFlow version: {tf.__version__}")print(f"- GPU disponível: {len(tf.config.list_physical_devices('GPU')) > 0}")


Note: you may need to restart the kernel to use updated packages.
✓ cnn_builder importado com sucesso!
✓ TensorFlow version: 2.20.0
✓ GPU disponível: False


## 2. Definir Parâmetros da Arquitetura CNN

Vamos definir os hiperparâmetros principais para construir nossa arquitetura CNN:

- **input_shape**: Dimensões da entrada (altura, largura, canais)
- **conv1_filters**: Número de filtros na primeira convolução (default: 32)
- **conv2_filters**: Número de filtros na segunda convolução (default: 64)  
- **kernel_size**: Tamanho do kernel/filtro (default: 3×3)
- **dense_units**: Número de neurônios na camada densa (default: 128)
- **dropout_rate**: Taxa de dropout para regularização (default: 0.5)

In [ ]:
# ==========================================# HIPERPARÂMETROS DA CNN - CONFIGURE A ENTRADA AQUI# ==========================================# DIMENSÕES DA ENTRADA - MUDE AQUI SE TIVER IMAGENS DIFERENTES!# Se suas imagens têm tamanho diferente, altere estes valores:INPUT_HEIGHT = 64       # Tamanho da altura (ex: 128, 256, 32)INPUT_WIDTH = 64        # Tamanho da largura (mesma altura/largura)INPUT_CHANNELS = 3      # Canais: 1 (cinza), 3 (RGB), 4 (RGBA)INPUT_SHAPE = (INPUT_HEIGHT, INPUT_WIDTH, INPUT_CHANNELS)# NÚMERO DE CLASSES - MUDE AQUI PARA SEU PROBLEMA# Quantas categorias diferentes você tem?N_CLASSES = 10  # Número de classes (ex: 2, 5, 100)# ==========================================# PARÂMETROS DAS CAMADAS CONVOLUCIONAIS# ==========================================# Primeira convoluçãoCONV1_FILTERS = 32          # Número de filtros (mais = mais features capturadas)KERNEL_SIZE_1 = (3, 3)     # Tamanho do kernel (3x3 é padrão, pode usar 5x5, 7x7)# Segunda convoluçãoCONV2_FILTERS = 64          # Mais filtros para features mais complexasKERNEL_SIZE_2 = (3, 3)     # Mesmo tamanho de kernel# ==========================================# PARÂMETROS DAS CAMADAS DE POOLING# ==========================================POOL_SIZE = (2, 2)          # Reduz dimensões pela metadePOOL_STRIDES = 2            # Deslocamento do pool# ==========================================# PARÂMETROS DAS CAMADAS DENSAS# ==========================================DENSE_UNITS = 128           # Neurônios na camada oculta (mais = mais complexidade)DROPOUT_RATE = 0.5          # Taxa de dropout (50% - bom valor padrão)# ==========================================# PARÂMETROS DE TREINAMENTO# ==========================================EPOCHS = 50BATCH_SIZE = 32VALIDATION_SPLIT = 0.2LEARNING_RATE = 0.001# Exibir resumo dos parâmetrosprint("======================================================================")print("PARÂMETROS DA ARQUITETURA CNN")print("======================================================================")print(f"\nENTRADA:")print(f"   Input Shape: {INPUT_SHAPE}")print(f"   Total de pixels por imagem: {INPUT_HEIGHT * INPUT_WIDTH * INPUT_CHANNELS:,}")print(f"   Classes: {N_CLASSES}")print(f"\nCONVOLUÇÃO 1:")print(f"   Filtros: {CONV1_FILTERS}")print(f"   Kernel: {KERNEL_SIZE_1}")print(f"\nCONVOLUÇÃO 2:")print(f"   Filtros: {CONV2_FILTERS}")print(f"   Kernel: {KERNEL_SIZE_2}")print(f"\nPOOLING:")print(f"   Pool Size: {POOL_SIZE}")print(f"   Strides: {POOL_STRIDES}")print(f"\nDENSE & DROPOUT:")print(f"   Dense Units: {DENSE_UNITS}")print(f"   Dropout Rate: {DROPOUT_RATE * 100}%")print(f"\nTREINAMENTO:")print(f"   Epochs: {EPOCHS}")print(f"   Batch Size: {BATCH_SIZE}")print(f"   Learning Rate: {LEARNING_RATE}")


╔════════════════════════════════════════════════════════════╗
║          PARÂMETROS DA ARQUITETURA CNN                     ║
╚════════════════════════════════════════════════════════════╝

📊 ENTRADA:
   Input Shape: (64, 64, 3)
   Total de pixels por imagem: 12,288
   Classes: 10

🔳 CONVOLUÇÃO 1:
   Filtros: 32
   Kernel: (3, 3)

🔳 CONVOLUÇÃO 2:
   Filtros: 64
   Kernel: (3, 3)

🔲 POOLING:
   Pool Size: (2, 2)
   Strides: 2

⬜ DENSE & DROPOUT:
   Dense Units: 128
   Dropout Rate: 50.0%

🎓 TREINAMENTO:
   Epochs: 50
   Batch Size: 32
   Learning Rate: 0.001


## 3. Construir Camadas Conv2D com Filtros e Kernels

A primeira camada convolucional extrai features básicas (bordas, texturas) usando 32 filtros 3×3 com ativação ReLU.
A segunda camada convolucional captura features mais complexas usando 64 filtros.

## 2.5 Como Usar Dados Reais (Entrada Customizada)

⚠️ **IMPORTANTE**: O notebook atual é educacional e não usa dados reais. 
Para treinar o modelo com seus próprios dados, siga o exemplo abaixo.


In [ ]:
# ==============================================================================═# EXEMPLO: COMO CARREGAR SEUS DADOS REAIS# ==============================================================================═# # DESCOMENTE O MÉTODO QUE CORRESPONDE AOS SEUS DADOS E EXECUTE ESTA CÉLULA## ──────────────────────────────────────────────────────────────────────────────# OPÇÃO 1: Dados de imagens em arquivos (JPG, PNG, etc)# ──────────────────────────────────────────────────────────────────────────────"""from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_arrayimport os# Configurar caminho das imagens# ⚠️ MUDE: caminho da pasta com suas imagens organizadas em subpastas por classedata_directory = "caminho/para/seus/dados"  # ← MUDE AQUI!#   Exemplo: "C:/datasets/meu_dataset/"#   Deve ter estrutura:#   data_directory/#   ├── classe_1/#   │   ├── img1.jpg#   │   ├── img2.jpg#   │   └── ...#   ├── classe_2/#   │   ├── img1.jpg#   │   └── ...# Carregar imagens automaticamentedata_generator = ImageDataGenerator(    rescale=1./255,                  # Normalizar para [0, 1]    validation_split=VALIDATION_SPLIT,    rotation_range=20,    width_shift_range=0.2,    height_shift_range=0.2,    shear_range=0.2,    zoom_range=0.2,    horizontal_flip=True,    fill_mode='nearest')# Dados DE TREINAMENTOtrain_data = data_generator.flow_from_directory(    data_directory,    target_size=(INPUT_HEIGHT, INPUT_WIDTH),  # ← Usa os parâmetros definidos acima!    batch_size=BATCH_SIZE,    class_mode='categorical',  # Para multiclassificação    subset='training')# Dados DE VALIDAÇÃOvalidation_data = data_generator.flow_from_directory(    data_directory,    target_size=(INPUT_HEIGHT, INPUT_WIDTH),    batch_size=BATCH_SIZE,    class_mode='categorical',    subset='validation')print("- Dados carregados com sucesso!")print(f"  Classes encontradas: {train_data.class_indices}")# Treinar o modelohistory = model.fit(    train_data,    epochs=EPOCHS,    validation_data=validation_data,    verbose=1)"""# ──────────────────────────────────────────────────────────────────────────────# OPÇÃO 2: Dados já em memória (numpy arrays)# ──────────────────────────────────────────────────────────────────────────────"""# Se você já tem seus dados em arrays NumPy:# ⚠️ MUDE: X_train, y_train com seus dados reaisX_train = np.random.random((100, INPUT_HEIGHT, INPUT_WIDTH, INPUT_CHANNELS))  # ← Dados de entraday_train = np.random.randint(0, N_CLASSES, 100)  # ← Rótulos (0 a N_CLASSES-1)# NormalizasX_train = X_train.astype('float32') / 255.0# Converter rótulos para one-hot encodingfrom tensorflow.keras.utils import to_categoricaly_train = to_categorical(y_train, N_CLASSES)print(f"Dados carregados: {X_train.shape}")print(f"Rótulos: {y_train.shape}")# Treinar o modelohistory = model.fit(    X_train, y_train,    epochs=EPOCHS,    batch_size=BATCH_SIZE,    validation_split=VALIDATION_SPLIT,    verbose=1)"""# ──────────────────────────────────────────────────────────────────────────────# OPÇÃO 3: Dados de um arquivo H5 (HDF5)# ──────────────────────────────────────────────────────────────────────────────"""import h5py# Arquivo HDF5 com seus dados# ⚠️ MUDE: caminho do arquivo e nomes das chavesh5_file = "caminho/para/seu_arquivo.h5"  # ← MUDE AQUI!with h5py.File(h5_file, 'r') as f:    # Nomes das chaves no arquivo (você precisa saber)    X_train = f['X_train'][:]  # ← MUDE O NOME DA CHAVE SE NECESSÁRIO    y_train = f['y_train'][:]        # Normalizar    X_train = X_train.astype('float32') / 255.0        # One-hot encoding se necessário    if len(y_train.shape) == 1:        from tensorflow.keras.utils import to_categorical        y_train = to_categorical(y_train, N_CLASSES)        print(f"Dados carregados: {X_train.shape}")        # Treinar    history = model.fit(        X_train, y_train,        epochs=EPOCHS,        batch_size=BATCH_SIZE,        validation_split=VALIDATION_SPLIT,        verbose=1    )"""# ──────────────────────────────────────────────────────────────────────────────# OPÇÃO 4: Dataset do Keras (CIFAR10, ImageNet, etc)# ──────────────────────────────────────────────────────────────────────────────"""from tensorflow.keras.datasets import cifar10# Carregar dataset público(X_train, y_train), (X_test, y_test) = cifar10.load_data()# NormalizarX_train = X_train.astype('float32') / 255.0X_test = X_test.astype('float32') / 255.0# Redimensionar se necessáriofrom tensorflow.image import resizeX_train = resize(X_train, (INPUT_HEIGHT, INPUT_WIDTH)).numpy()X_test = resize(X_test, (INPUT_HEIGHT, INPUT_WIDTH)).numpy()# One-hot encodingfrom tensorflow.keras.utils import to_categoricaly_train = to_categorical(y_train, N_CLASSES)y_test = to_categorical(y_test, N_CLASSES)print(f"Dados carregados: {X_train.shape}")# Treinarhistory = model.fit(    X_train, y_train,    epochs=EPOCHS,    batch_size=BATCH_SIZE,    validation_data=(X_test, y_test),    verbose=1)"""print("\n" + "="*80)print(" RESUMO - COMO USAR SEUS DADOS:")print("="*80)print("""1. MODIFIQUE os parâmetros acima (INPUT_HEIGHT, INPUT_WIDTH, INPUT_CHANNELS, N_CLASSES)2. ESCOLHA uma das 4 opções de carregamento de dados3. DESCOMENTE a opção escolhida4. MUDE os caminhos/nomes para seus arquivos/pastas5. EXECUTE a célula6. O modelo será treinado automaticamente!Formatos esperados:- X_train: shape (num_amostras, INPUT_HEIGHT, INPUT_WIDTH, INPUT_CHANNELS)- y_train: shape (num_amostras, N_CLASSES) [one-hot] ou (num_amostras,) [inteiros 0 a N_CLASSES-1]- Valores deve estar NORMALIZADOS entre 0 e 1!""")


In [ ]:
# Explicação das camadas Conv2Dprint("CAMADA CONV2D - Explicação Técnica\n")print("="*70)print("\nPRIMEIRA CONVOLUÇÃO (Conv2D_1):")print(f"   Filtros: {CONV1_FILTERS}")print(f"   Kernel: {KERNEL_SIZE_1} (matriz 3x3)")print(f"   Padding: 'same' (mantém dimensões)")print(f"   Ativação: ReLU (f(x) = max(0, x))")print(f"   Parâmetros: 3x3x3x{CONV1_FILTERS} + {CONV1_FILTERS} = {3*3*3*CONV1_FILTERS + CONV1_FILTERS:,}")print(f"\n   Função: Extrair features básicas (bordas, texturas)")print(f"   Output shape: ({INPUT_HEIGHT}, {INPUT_WIDTH}, {CONV1_FILTERS})")print("\n\nSEGUNDA CONVOLUÇÃO (Conv2D_2):")print(f"   Filtros: {CONV2_FILTERS}")print(f"   Kernel: {KERNEL_SIZE_2}")print(f"   Padding: 'same'")print(f"   Ativação: ReLU")print(f"   Parâmetros: 3x3x{CONV1_FILTERS}x{CONV2_FILTERS} + {CONV2_FILTERS} = {3*3*CONV1_FILTERS*CONV2_FILTERS + CONV2_FILTERS:,}")print(f"\n   Função: Extrair features mais complexas")print(f"   Output shape: (16, 16, {CONV2_FILTERS})  [após pooling]")print("\n" + "="*70)print("\nPor que kernel 3x3?")print("   - Tamanho padrão em deep learning moderno")print("   - Captura padrões locais eficientemente")print("   - Menos parâmetros que kernels 5x5 ou 7x7")print("   - Dois kernels 3x3 = um kernel 5x5 (mesma receptive field)")print("\nPor que ReLU no lugar de Sigmoid?")print("   - Convergência mais rápida")print("   - Evita problema de \"vanishing gradient\"")print("   - Promove esparsidade (alguns neurônios inativos)")print("   - Computacionalmente mais eficiente")


🔍 CAMADA CONV2D - Explicação Técnica


📐 PRIMEIRA CONVOLUÇÃO (Conv2D_1):
   • Filtros: 32
   • Kernel: (3, 3) (matriz 3×3)
   • Padding: 'same' (mantém dimensões)
   • Ativação: ReLU (f(x) = max(0, x))
   • Parâmetros: 3×3×3×32 + 32 = 896

   Função: Extrair features básicas (bordas, texturas)
   Output shape: (64, 64, 32)


📐 SEGUNDA CONVOLUÇÃO (Conv2D_2):
   • Filtros: 64
   • Kernel: (3, 3)
   • Padding: 'same'
   • Ativação: ReLU
   • Parâmetros: 3×3×32×64 + 64 = 18,496

   Função: Extrair features mais complexas
   Output shape: (16, 16, 64)  [após pooling]


💡 Por que kernel 3×3?
   ✓ Tamanho padrão em deep learning moderno
   ✓ Captura padrões locais eficientemente
   ✓ Menos parâmetros que kernels 5×5 ou 7×7
   ✓ Dois kernels 3×3 = um kernel 5×5 (mesma receptive field)

💡 Por que ReLU no lugar de Sigmoid?
   ✓ Convergência mais rápida
   ✓ Evita problema de "vanishing gradient"
   ✓ Promove esparsidade (alguns neurônios inativos)
   ✓ Computacionalmente mais eficiente


## 4. Adicionar Camadas MaxPooling

As camadas MaxPooling2D reduzem as dimensões espaciais pela metade, capturando as features mais importantes.
Isso reduz o número de parâmetros e melhora a robustez das features extraídas.

In [9]:
# Explicação das camadas MaxPoolingprint("\n🔄 CAMADA MAXPOOLING - Explicação Técnica\n")print("="*70)# Calcular dimensões após cada poolingheight_after_pool1 = INPUT_HEIGHT // 2width_after_pool1 = INPUT_WIDTH // 2height_after_pool2 = height_after_pool1 // 2width_after_pool2 = width_after_pool1 // 2print("\nPRIMEIRA MAXPOOLING (MaxPooling2D_1):")print(f"   • Pool Size: {POOL_SIZE}")print(f"   • Strides: {POOL_STRIDES}")print(f"   • Função: Selecionar o valor máximo em cada janela 2x2")print(f"   • Input shape: ({INPUT_HEIGHT}, {INPUT_WIDTH}, {CONV1_FILTERS})")print(f"   • Output shape: ({height_after_pool1}, {width_after_pool1}, {CONV1_FILTERS})")print(f"   • Redução: 4x menos parâmetros ({(INPUT_HEIGHT*INPUT_WIDTH)/(height_after_pool1*width_after_pool1):.0f}x total)")print("\nSEGUNDA MAXPOOLING (MaxPooling2D_2):")print(f"   • Pool Size: {POOL_SIZE}")print(f"   • Strides: {POOL_STRIDES}")print(f"   • Input shape: ({height_after_pool1}, {width_after_pool1}, {CONV2_FILTERS})")print(f"   • Output shape: ({height_after_pool2}, {width_after_pool2}, {CONV2_FILTERS})")total_pixels_before = INPUT_HEIGHT * INPUT_WIDTHtotal_pixels_after = height_after_pool2 * width_after_pool2reduction_factor = total_pixels_before / total_pixels_afterprint(f"\n📈 REDUÇÃO ACUMULATIVA:")print(f"   • Pixels iniciais: {total_pixels_before:,}")print(f"   • Pixels finais: {total_pixels_after:,}")print(f"   • Fator de redução: {reduction_factor:.1f}x")print("\n" + "="*70)print("\n Benefícios do MaxPooling:")print(f"   - Reduz dimensionalidade (de {INPUT_HEIGHT*INPUT_WIDTH:,} para {total_pixels_after:,} pixels)")print(f"   - Seleção de features mais relevantes (max = mais ativado)")print(f"   - Aumenta receptive field (visão mais ampla)")print(f"   - Invariância a pequenas translações")print(f"   - Regularização implícita (evita overfitting)")print(f"   - Reduz custo computacional")# Visualizar exemplo de MaxPoolingprint("\n\n EXEMPLO VISUAL - MaxPooling 2x2:")print("""Input (4x4):              MaxPooling 2x2:[1  2  3  4]             [2  4][5  6  7  8]      →      [14 16][9  10 11 12][13 14 15 16]""")print("Funcionamento:")print("   Bloco 1 (superior-esquerdo): max(1,2,5,6) = 6 → 2")print("   Bloco 2 (superior-direito):  max(3,4,7,8) = 8 → 4")print("   Bloco 3 (inferior-esquerdo): max(9,10,13,14) = 14 → 14")print("   Bloco 4 (inferior-direito):  max(11,12,15,16) = 16 → 16")



🔄 CAMADA MAXPOOLING - Explicação Técnica


📊 PRIMEIRA MAXPOOLING (MaxPooling2D_1):
   • Pool Size: (2, 2)
   • Strides: 2
   • Função: Selecionar o valor máximo em cada janela 2×2
   • Input shape: (64, 64, 32)
   • Output shape: (32, 32, 32)
   • Redução: 4× menos parâmetros (4× total)

📊 SEGUNDA MAXPOOLING (MaxPooling2D_2):
   • Pool Size: (2, 2)
   • Strides: 2
   • Input shape: (32, 32, 64)
   • Output shape: (16, 16, 64)

📈 REDUÇÃO ACUMULATIVA:
   • Pixels iniciais: 4,096
   • Pixels finais: 256
   • Fator de redução: 16.0×


💡 Benefícios do MaxPooling:
   ✓ Reduz dimensionalidade (de 4,096 para 256 pixels)
   ✓ Seleção de features mais relevantes (max = mais ativado)
   ✓ Aumenta receptive field (visão mais ampla)
   ✓ Invariância a pequenas translações
   ✓ Regularização implícita (evita overfitting)
   ✓ Reduz custo computacional


🎨 EXEMPLO VISUAL - MaxPooling 2×2:

Input (4×4):              MaxPooling 2×2:
[1  2  3  4]             [2  4]
[5  6  7  8]      →      [14 16]
[9  

## 5. Camadas Flatten e Dense

Após o processamento convolucional e pooling, precisamos achatar (flatten) a representação multidimensional
em um vetor 1D para alimentar as camadas fully connected (Dense).

In [10]:
# Explicação das camadas Flatten e Denseprint("\n CAMADA FLATTEN - Explicação Técnica\n")print("="*70)flatten_input_size = height_after_pool2 * width_after_pool2 * CONV2_FILTERSprint(f"\nFLATTEN LAYER:")print(f"   • Input shape: ({height_after_pool2}, {width_after_pool2}, {CONV2_FILTERS})")print(f"   • Output shape: ({flatten_input_size:,},)")print(f"   • Operação: Converter tensor 4D em vetor 1D")print(f"   • Parâmetros: 0 (apenas reorganiza dados)")print("\n Função do Flatten:")print("   - Necessário para conectar camadas convolucionais com Dense")print("   - Converte (16, 16, 64) em um vetor de 16,384 elementos")print("   - Mantém informação, apenas reorganiza")print("\n\nCAMADA DENSE - Explicação Técnica\n")print("="*70)print(f"\nDENSE LAYER (Oculta):")print(f"   • Input features: {flatten_input_size:,}")print(f"   • Unidades: {DENSE_UNITS}")print(f"   • Ativação: ReLU")print(f"   • Parâmetros: {flatten_input_size} x {DENSE_UNITS} + {DENSE_UNITS} = {flatten_input_size * DENSE_UNITS + DENSE_UNITS:,}")print(f"\n Função da Dense Oculta:")print("   - Aprender representações não-lineares")print("   - Captar correlações globais entre features")print("   - Reduzir dimensionalidade (16,384 → 128)")print(f"\nDROPOUT LAYER:")print(f"   • Taxa: {DROPOUT_RATE * 100}%")print(f"   • Função: Desativar aleatoriamente {int(DENSE_UNITS * DROPOUT_RATE)} neurônios durante treinamento")print(f"   • Benefício: Reduz overfitting, melhora generalização")print(f"   • Parâmetros: 0 (técnica de regularização)")print(f"\nDENSE LAYER (Saída):")print(f"   • Input features: {DENSE_UNITS}")print(f"   • Unidades (classes): {N_CLASSES}")print(f"   • Ativação: Softmax (multiclasse)")print(f"   • Parâmetros: {DENSE_UNITS} x {N_CLASSES} + {N_CLASSES} = {DENSE_UNITS * N_CLASSES + N_CLASSES:,}")print(f"\n Função da Dense Saída:")print("   - Produzir distribuição de probabilidade")print("   - Softmax normaliza saídas (soma = 1.0)")print("   - Conecta 128 features aos 10 rótulos")# Visualizar transformação de shapesprint("\n\n TRANSFORMAÇÃO DE SHAPES:")print("="*70)print(f"Input:              ({INPUT_HEIGHT}, {INPUT_WIDTH}, {INPUT_CHANNELS})")print(f"After Conv2D_1:     ({INPUT_HEIGHT}, {INPUT_WIDTH}, {CONV1_FILTERS})")print(f"After MaxPool_1:    ({height_after_pool1}, {width_after_pool1}, {CONV1_FILTERS})")print(f"After Conv2D_2:     ({height_after_pool1}, {width_after_pool1}, {CONV2_FILTERS})")print(f"After MaxPool_2:    ({height_after_pool2}, {width_after_pool2}, {CONV2_FILTERS})")print(f"After Flatten:      ({flatten_input_size:,},)")print(f"After Dense_hidden: ({DENSE_UNITS},)")print(f"After Dropout:      ({DENSE_UNITS},)  [sem mudança de shape]")print(f"Output (Softmax):   ({N_CLASSES},)    [distribuição de probs]")



🔀 CAMADA FLATTEN - Explicação Técnica


📊 FLATTEN LAYER:
   • Input shape: (16, 16, 64)
   • Output shape: (16,384,)
   • Operação: Converter tensor 4D em vetor 1D
   • Parâmetros: 0 (apenas reorganiza dados)

💡 Função do Flatten:
   ✓ Necessário para conectar camadas convolucionais com Dense
   ✓ Converte (16, 16, 64) em um vetor de 16,384 elementos
   ✓ Mantém informação, apenas reorganiza


⬜ CAMADA DENSE - Explicação Técnica


📊 DENSE LAYER (Oculta):
   • Input features: 16,384
   • Unidades: 128
   • Ativação: ReLU
   • Parâmetros: 16384 × 128 + 128 = 2,097,280

💡 Função da Dense Oculta:
   ✓ Aprender representações não-lineares
   ✓ Captar correlações globais entre features
   ✓ Reduzir dimensionalidade (16,384 → 128)

📊 DROPOUT LAYER:
   • Taxa: 50.0%
   • Função: Desativar aleatoriamente 64 neurônios durante treinamento
   • Benefício: Reduz overfitting, melhora generalização
   • Parâmetros: 0 (técnica de regularização)

📊 DENSE LAYER (Saída):
   • Input features: 128
   • Un

## 6. Criar Modelo CNN Completo

Agora vamos criar o modelo CNN completo usando a função `build_cnn_model()` do módulo `cnn_builder.py`.
A arquitetura segue o padrão: Input → Conv2D → MaxPooling → Conv2D → MaxPooling → Flatten → Dense → Output

In [11]:
print("\n CONSTRUINDO MODELO CNN...\n")# Criar o modelo usando a função do cnn_buildermodel = build_cnn_model(    input_shape=INPUT_SHAPE,    n_classes=N_CLASSES,    conv1_filters=CONV1_FILTERS,    conv2_filters=CONV2_FILTERS,    kernel_size=(3, 3),    dense_units=DENSE_UNITS,    dropout_rate=DROPOUT_RATE)print("- Modelo construído com sucesso!")print(f"\nInformações do Modelo:")print(f"   • Type: {type(model).__name__}")print(f"   • Total de camadas: {len(model.layers)}")print(f"   • Status compilação: {'- Compilado' if model.optimizer else '✗ Não compilado'}")# Exibir tipos de camadaslayer_types = {}for layer in model.layers:    layer_name = layer.__class__.__name__    layer_types[layer_name] = layer_types.get(layer_name, 0) + 1print(f"\nTipos de Camadas:")for layer_type, count in sorted(layer_types.items()):    print(f"   • {layer_type}: {count}")# Exibir nomes das camadasprint(f"\nDetalhes das Camadas:")for i, layer in enumerate(model.layers, 1):    print(f"   {i}. {layer.name:20s} → {layer.__class__.__name__:15s} (params: {layer.count_params():>10,})")



🏗️  CONSTRUINDO MODELO CNN...

✓ Modelo construído com sucesso!

📊 Informações do Modelo:
   • Type: Sequential
   • Total de camadas: 8
   • Status compilação: ✓ Compilado

📋 Tipos de Camadas:
   • Conv2D: 2
   • Dense: 2
   • Dropout: 1
   • Flatten: 1
   • MaxPooling2D: 2

🔗 Detalhes das Camadas:
   1. conv2d_1             → Conv2D          (params:        896)
   2. maxpooling2d_1       → MaxPooling2D    (params:          0)
   3. conv2d_2             → Conv2D          (params:     18,496)
   4. maxpooling2d_2       → MaxPooling2D    (params:          0)
   5. flatten              → Flatten         (params:          0)
   6. dense_hidden         → Dense           (params:  2,097,280)
   7. dropout              → Dropout         (params:          0)
   8. output               → Dense           (params:      1,290)


## 7. Compilar e Resumir Modelo

Vamos compilar o modelo com Adam optimizer e exibir um resumo completo da arquitetura.

In [12]:
print("\n" + "="*80)print("RESUMO COMPLETO DO MODELO CNN")print("="*80 + "\n")# Exibir model.summary() com mais detalhesmodel.summary()# Obter e exibir informações de arquiteturaprint("\n" + "="*80)print("ESTATÍSTICAS DE PARÂMETROS")print("="*80 + "\n")architecture_summary = get_model_architecture_summary(model)print(f"Total de Parâmetros:       {architecture_summary['total_params']:>15,}")print(f"Parâmetros Treináveis:     {architecture_summary['trainable_params']:>15,}")print(f"Parâmetros Não-treináveis: {architecture_summary['non_trainable_params']:>15,}")# Percentualtrainable_pct = (architecture_summary['trainable_params'] / architecture_summary['total_params']) * 100print(f"\nTreinável: {trainable_pct:.1f}% dos parâmetros")# Estimar tamanho em MBsize_mb = (architecture_summary['total_params'] * 4) / (1024 ** 2)  # 4 bytes por float32print(f"Tamanho estimado (float32): {size_mb:.2f} MB")# Parâmetros por camadaprint(f"\n" + "="*80)print("PARÂMETROS POR CAMADA")print("="*80 + "\n")print(f"{'Camada':<20} {'Tipo':<15} {'Output Shape':<25} {'Parâmetros':>15}")print("-" * 80)for layer_info in architecture_summary['layers_info']:    print(f"{layer_info['name']:<20} "          f"{layer_info['type']:<15} "          f"{str(layer_info['output_shape']):<25} "          f"{layer_info['params']:>15,}")# Informações de compilaçãoprint(f"\n" + "="*80)print(" CONFIGURAÇÃO DE COMPILAÇÃO")print("="*80 + "\n")print(f"Otimizador:  {model.optimizer.__class__.__name__}")print(f"Loss:        {model.loss}")print(f"Métricas:    {', '.join(model.metrics_names) if model.metrics_names else 'Nenhuma'}")print(f"\n" + "="*80)print("Modelo CNN pronto para treinamento!")print("="*80)



📋 RESUMO COMPLETO DO MODELO CNN



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpooling2d_1 (MaxPooling2D)   │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpooling2d_2 (MaxPooling2D)   │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 16384)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_hidden (Dense)            │ (None, 128)            │     2,097,280 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 10)             │         1,290 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,117,962 (8.08 MB)

 Trainable params: 2,117,962 (8.08 MB)

 Non-trainable params: 0 (0.00 B)


📊 ESTATÍSTICAS DE PARÂMETROS

Total de Parâmetros:             2,117,962
Parâmetros Treináveis:           2,117,962
Parâmetros Não-treináveis:               0

Treinável: 100.0% dos parâmetros
Tamanho estimado (float32): 8.08 MB

📍 PARÂMETROS POR CAMADA

Camada               Tipo            Output Shape                   Parâmetros
--------------------------------------------------------------------------------
conv2d_1             Conv2D          (None, 64, 64, 32)                    896
maxpooling2d_1       MaxPooling2D    (None, 32, 32, 32)                      0
conv2d_2             Conv2D          (None, 32, 32, 64)                 18,496
maxpooling2d_2       MaxPooling2D    (None, 16, 16, 64)                      0
flatten              Flatten         (None, 16384)                           0
dense_hidden         Dense           (None, 128)                     2,097,280
dropout              Dropout         (None, 128)                             0
output               Dense     